# NER Evaluation Workflow — Template

Template showing the three-stage evaluation pattern from the [workflow documentation](../WORKFLOW.md).

Replace the placeholder values (`YOUR_MODEL_NAME`, `YOUR_DATASET`, etc.) with the actual model and dataset for your target language.

**Stages:**
1. Install and imports
2. Benchmark evaluation (Stage 2 of the workflow)
3. Extraction on project data (Stage 3)
4. Validation app launch

## Setup

**Google Colab:** uncomment and run the install cell below.  
**Local:** install with `pip install git+https://github.com/ay94/multilingual-ner.git`

In [ ]:
# Colab install — uncomment if running in Google Colab
# !pip install git+https://github.com/ay94/multilingual-ner.git transformers datasets seqeval

In [ ]:
import sys
sys.path.insert(0, '..')  # local dev — remove if installed via pip

import pandas as pd
from multilingual_ner.evaluation import ReadNERData, ModelEvaluation, align_dataset, check_labels
from multilingual_ner.extraction import NamedEntityExtractions

---
## Stage 1 — Model and Benchmark Identification

Before running this notebook, complete Stage 1 manually:

1. Search HuggingFace model hub for NER models trained on your target language
2. Check the model license and annotation scheme (does it use CoNLL BIO? What entity types?)
3. Find a benchmark dataset — check HuggingFace Datasets, leaderboards, or language-specific repositories
4. Confirm the benchmark test split was not used to train any of the candidate models

Fill in the values below before continuing.

In [ ]:
# --- Fill these in ---
LANGUAGE        = 'xx'                  # ISO language code, e.g. 'ar', 'zu', 'tr'
MODEL_NAME      = 'YOUR_MODEL_NAME'     # HuggingFace model ID
DATASET_NAME    = 'YOUR_DATASET'        # HuggingFace dataset ID, e.g. 'wikiann'
DATASET_LANG    = LANGUAGE              # language config for the dataset (if required)

# Label map for the benchmark dataset (integer IDs → string labels)
# Check the dataset card on HuggingFace for the correct mapping
DATASET_LABEL_MAP = {
    'O':     0,
    'B-PER': 1,
    'I-PER': 2,
    'B-ORG': 3,
    'I-ORG': 4,
    'B-LOC': 5,
    'I-LOC': 6,
}

---
## Stage 2 — Benchmark Evaluation

Load the benchmark dataset, check and align labels, load the model, run evaluation.

In [ ]:
# Load benchmark dataset
reader = ReadNERData()
words, labels = reader.read_dataset(DATASET_NAME, DATASET_LABEL_MAP, lang=DATASET_LANG)

print(f'Loaded {len(words)} sentences')
print('Labels in dataset:', check_labels(labels))

In [ ]:
# Load model and inspect its label scheme
# This tells you what alignment mapping to define below
model_eval = ModelEvaluation(MODEL_NAME)
print(model_eval.model.config.id2label)

In [ ]:
# Define label alignment: map model output labels → standard scheme (PER, LOC, ORG, MISC)
# Adjust based on what model.config.id2label printed above
MODEL_LABEL_ALIGNMENT = {
    'B-PER':  'B-PER',
    'I-PER':  'I-PER',
    'B-LOC':  'B-LOC',
    'I-LOC':  'I-LOC',
    'B-ORG':  'B-ORG',
    'I-ORG':  'I-ORG',
    'B-MISC': 'O',
    'I-MISC': 'O',
    'O':      'O',
    # Add any additional model-specific labels here
}

In [ ]:
# Run evaluation
model_eval = ModelEvaluation(MODEL_NAME, MODEL_LABEL_ALIGNMENT)
evaluation_output = model_eval.evaluate_model(words, labels)

In [ ]:
# Entity-level results (seqeval)
seqeval_results = evaluation_output.get_classification('Seqeval')
seqeval_results

In [ ]:
# Token-level results (sklearn)
sklearn_results = evaluation_output.get_classification('Sklearn')
sklearn_results

---
## Stage 3 — Extraction on Project Data

Apply the selected model to unlabelled project text. Replace the dummy sentences below with real project data.  
Required columns: `text`, `message_id`, `accountId`.

In [ ]:
# Dummy project data — replace with real data
project_data = pd.DataFrame({
    'text': [
        'Replace this with sentence 1 in your target language.',
        'Replace this with sentence 2 in your target language.',
        'Replace this with sentence 3 in your target language.',
    ],
    'message_id': ['msg_001', 'msg_002', 'msg_003'],
    'accountId':  ['acc_1',   'acc_1',   'acc_2'],
})

project_data.head()

In [ ]:
# Run extraction
extractor = NamedEntityExtractions(
    model_name=MODEL_NAME,
    project_data=project_data,
    text_col='text',
    batch_size=8,
)

json_schema, output_df, post_processed, raw_outputs = extractor.extract_outputs()

In [ ]:
# View structured output
output_df[['text', 'PER', 'LOC', 'ORG', 'MISC']]

In [ ]:
# View first JSON record
import json
print(json.dumps(json_schema[0], ensure_ascii=False, indent=2))

---
## Validation app

Launch the Dash app to qualitatively review extraction outputs:

```bash
python -m multilingual_ner.validation
# Opens at http://localhost:8050
```